## Task 1: Managing Conversation History with Summarization
1. Maintain a running conversation history of user–assistant chats.
2. Implement summarization of conversation history to keep it concise.
3. Allow customization options for truncation:
a. Limit by number of conversation turns (e.g., last n messages).
b. Limit by character/word length.

4. Add periodic summarization:
a. Perform summarization after every k-th run of the conversation.
b. Store/replace the summarized version in the conversation history.

5. Demonstrate the functionality by:
a. Feeding multiple conversation samples.
b. Showing outputs at different truncation settings.
c. Showing how summarization happens after every k-th run (e.g., after every 3rd run).

In [53]:
# Install required dependencies
%pip install openai

import json
import os
from typing import List, Dict, Optional, Union
from openai import OpenAI
import time


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Set your Groq API key here
GROQ_API_KEY = ""  # Replace with your actual API key


✅ API key is set!


## ConversationManager Implementation


In [55]:
class ConversationManager:
    """
    Manages conversation history with summarization capabilities using Groq API
    """
    
    def __init__(self, groq_api_key: str, model: str = "llama-3.1-8b-instant"):
        """
        Initialize the conversation manager
        
        Args:
            groq_api_key: Groq API key
            model: Model to use for summarization
        """
        self.client = OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=groq_api_key
        )
        self.model = model
        self.conversation_history: List[Dict[str, str]] = []
        self.run_count = 0
        self.summary = ""
        
    def add_message(self, role: str, content: str):
        """
        Add a message to conversation history
        
        Args:
            role: 'user' or 'assistant'
            content: Message content
        """
        message = {"role": role, "content": content}
        self.conversation_history.append(message)
        
    def get_conversation_length(self) -> Dict[str, int]:
        """
        Get conversation statistics
        
        Returns:
            Dictionary with turns count and character count
        """
        turns = len(self.conversation_history)
        total_chars = sum(len(msg["content"]) for msg in self.conversation_history)
        total_words = sum(len(msg["content"].split()) for msg in self.conversation_history)
        
        return {
            "turns": turns,
            "characters": total_chars,
            "words": total_words
        }
    
    def truncate_by_turns(self, max_turns: int) -> List[Dict[str, str]]:
        """
        Truncate conversation to last n turns
        
        Args:
            max_turns: Maximum number of turns to keep
            
        Returns:
            Truncated conversation history
        """
        if max_turns <= 0:
            return []
        return self.conversation_history[-max_turns:]
    
    def truncate_by_length(self, max_chars: Optional[int] = None, max_words: Optional[int] = None) -> List[Dict[str, str]]:
        """
        Truncate conversation by character or word limit
        
        Args:
            max_chars: Maximum character count
            max_words: Maximum word count
            
        Returns:
            Truncated conversation history
        """
        if not max_chars and not max_words:
            return self.conversation_history
            
        truncated = []
        current_chars = 0
        current_words = 0
        
        # Start from the end and work backwards
        for message in reversed(self.conversation_history):
            msg_chars = len(message["content"])
            msg_words = len(message["content"].split())
            
            # Check if adding this message would exceed limits
            if max_chars and (current_chars + msg_chars > max_chars):
                break
            if max_words and (current_words + msg_words > max_words):
                break
                
            truncated.insert(0, message)
            current_chars += msg_chars
            current_words += msg_words
            
        return truncated
    
    def summarize_conversation(self, messages: List[Dict[str, str]]) -> str:
        """
        Summarize a list of conversation messages using Groq API
        
        Args:
            messages: List of conversation messages
            
        Returns:
            Summary string
        """
        if not messages:
            return ""
            
        # Format conversation for summarization
        conversation_text = "\n".join([
            f"{msg['role'].upper()}: {msg['content']}" 
            for msg in messages
        ])
        
        system_prompt = """You are a helpful assistant that summarizes conversations concisely. 
       Summarize the following conversation history in a clear, short, and concise way. 
       Capture only the key points, main actions, and decisions, while removing unnecessary details or repetition. 
       Keep it under a few sentences."""
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Please summarize this conversation:\n\n{conversation_text}"}
                ],
                temperature=0.3,
                max_tokens=500
            )
            
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            print(f"Error during summarization: {e}")
            return f"Summary unavailable due to error: {str(e)}"
    
    def periodic_summarization(self, k_runs: int):
        """
        Perform periodic summarization after every k runs
        
        Args:
            k_runs: Number of runs after which to summarize
        """
        self.run_count += 1
        
        if self.run_count % k_runs == 0 and self.conversation_history:
            print(f"\n--- Performing periodic summarization (Run #{self.run_count}) ---")
            
            # Summarize current conversation
            new_summary = self.summarize_conversation(self.conversation_history)
            
            # Combine with existing summary if available
            if self.summary:
                combined_text = f"Previous summary: {self.summary}\n\nRecent conversation: {new_summary}"
                self.summary = self.summarize_conversation([
                    {"role": "system", "content": combined_text}
                ])
            else:
                self.summary = new_summary
            
            # Replace conversation history with summary
            self.conversation_history = [
                {"role": "system", "content": f"Conversation Summary: {self.summary}"}
            ]
            
            print(f"Conversation summarized and replaced. New summary length: {len(self.summary)} characters")
    
    def get_context_for_response(self, max_turns: Optional[int] = None, 
                                max_chars: Optional[int] = None, 
                                max_words: Optional[int] = None) -> List[Dict[str, str]]:
        """
        Get conversation context with optional truncation
        
        Args:
            max_turns: Maximum number of turns
            max_chars: Maximum character count
            max_words: Maximum word count
            
        Returns:
            Conversation context for API calls
        """
        context = self.conversation_history
        
        # Apply truncation if specified
        if max_turns:
            context = self.truncate_by_turns(max_turns)
        elif max_chars or max_words:
            context = self.truncate_by_length(max_chars, max_words)
            
        return context
    
    def display_conversation_stats(self):
        """Display current conversation statistics"""
        stats = self.get_conversation_length()
        print(f"\n--- Conversation Statistics ---")
        print(f"Total turns: {stats['turns']}")
        print(f"Total characters: {stats['characters']}")
        print(f"Total words: {stats['words']}")
        print(f"Run count: {self.run_count}")
        if self.summary:
            print(f"Current summary: {self.summary[:100]}...")
        print("-" * 30)
    
    def display_conversation(self, max_display: int = 5):
        """
        Display recent conversation messages
        
        Args:
            max_display: Maximum number of recent messages to display
        """
        print(f"\n--- Recent Conversation (last {max_display} messages) ---")
        recent_messages = self.conversation_history[-max_display:]
        
        for i, msg in enumerate(recent_messages, 1):
            role = msg["role"].upper()
            content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
            print(f"{i}. {role}: {content}")
        print("-" * 50)

print("✅ ConversationManager class loaded successfully!")


✅ ConversationManager class loaded successfully!



Let's demonstrate all the features of our conversation management system.


In [56]:
# Initialize the conversation manager
if GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠️  Please set your GROQ_API_KEY first!")
else:
    manager = ConversationManager(GROQ_API_KEY)
    print("🚀 ConversationManager initialized with Groq API!")
    print(f"Using model: {manager.model}")


🚀 ConversationManager initialized with Groq API!
Using model: llama-3.1-8b-instant


### 1. Adding Sample Conversation Data


In [57]:
# Sample conversation about machine learning and data science
sample_conversations = [
    ("user", "I'm starting a machine learning project to predict house prices. What approach should I take?"),
    ("assistant", "Great project! For house price prediction, you'll want to use regression models. Start by gathering features like location, size, number of bedrooms/bathrooms, age, and local amenities. Linear regression is a good baseline, but you might get better results with Random Forest or XGBoost."),
    ("user", "I have a dataset with 50,000 houses and 20 features. How should I handle missing values?"),
    ("assistant", "With that size dataset, you have several options: 1) For numerical features, use median imputation or KNN imputation, 2) For categorical features, use mode imputation or create a 'missing' category, 3) Consider dropping features with >30% missing values, 4) Use algorithms like XGBoost that handle missing values naturally."),
    ("user", "What about feature engineering? Are there specific techniques for real estate data?"),
    ("assistant", "Absolutely! For real estate: 1) Create price per square foot, 2) Calculate distance to city center/schools/hospitals, 3) Extract neighborhood crime rates and school ratings, 4) Create age bins (new/mid-age/old), 5) Seasonal factors (month of sale), 6) Polynomial features for size relationships."),
    ("user", "How do I evaluate my model's performance and avoid overfitting?"),
    ("assistant", "Use these evaluation strategies: 1) Split data: 70% train, 15% validation, 15% test, 2) Metrics: RMSE, MAE, and R² score, 3) Cross-validation with 5-10 folds, 4) Learning curves to detect overfitting, 5) Regularization (L1/L2) for linear models, 6) Early stopping for tree-based models."),
    ("user", "Should I try deep learning for this problem?"),
    ("assistant", "For tabular data like house prices, traditional ML often outperforms deep learning. Try gradient boosting (XGBoost, LightGBM) first - they're easier to interpret and usually perform better on structured data. Deep learning might help if you incorporate image data (house photos) or have very complex feature interactions."),
    ("user", "How can I make my model interpretable for stakeholders?"),
    ("assistant", "Use these interpretability techniques: 1) Feature importance from tree-based models, 2) SHAP values for individual predictions, 3) Partial dependence plots, 4) LIME for local explanations, 5) Simple correlation analysis, 6) Create business-friendly dashboards showing key factors affecting price predictions.")
]

print("📝 Adding conversation messages...")
for i, (role, content) in enumerate(sample_conversations, 1):
    if GROQ_API_KEY != "your_groq_api_key_here":
        manager.add_message(role, content)
        print(f"{i}. Added {role} message ({len(content)} chars)")

if GROQ_API_KEY != "your_groq_api_key_here":
    print("\\n✅ All messages added!")
    manager.display_conversation_stats()


📝 Adding conversation messages...
1. Added user message (93 chars)
2. Added assistant message (287 chars)
3. Added user message (88 chars)
4. Added assistant message (323 chars)
5. Added user message (83 chars)
6. Added assistant message (295 chars)
7. Added user message (63 chars)
8. Added assistant message (286 chars)
9. Added user message (44 chars)
10. Added assistant message (321 chars)
11. Added user message (55 chars)
12. Added assistant message (308 chars)
\n✅ All messages added!

--- Conversation Statistics ---
Total turns: 12
Total characters: 2246
Total words: 331
Run count: 0
------------------------------


### 2. Demonstrating Truncation Options


In [58]:
print("🔄 Testing Truncation by Number of Turns")
print("=" * 45)

if GROQ_API_KEY != "your_groq_api_key_here":
    # Test different turn limits
    for turns in [3, 5, 8]:
        truncated = manager.truncate_by_turns(turns)
        print(f"\\n📊 Last {turns} turns ({len(truncated)} messages):")
        
        for i, msg in enumerate(truncated, 1):
            content_preview = msg['content'][:60] + "..." if len(msg['content']) > 60 else msg['content']
            print(f"  {i}. {msg['role'].upper()}: {content_preview}")
        
        total_chars = sum(len(msg['content']) for msg in truncated)
        print(f"  📈 Total characters: {total_chars}")
else:
    print("⚠️  Please set your GROQ_API_KEY to test truncation!")


🔄 Testing Truncation by Number of Turns
\n📊 Last 3 turns (3 messages):
  1. ASSISTANT: For tabular data like house prices, traditional ML often out...
  2. USER: How can I make my model interpretable for stakeholders?
  3. ASSISTANT: Use these interpretability techniques: 1) Feature importance...
  📈 Total characters: 684
\n📊 Last 5 turns (5 messages):
  1. ASSISTANT: Use these evaluation strategies: 1) Split data: 70% train, 1...
  2. USER: Should I try deep learning for this problem?
  3. ASSISTANT: For tabular data like house prices, traditional ML often out...
  4. USER: How can I make my model interpretable for stakeholders?
  5. ASSISTANT: Use these interpretability techniques: 1) Feature importance...
  📈 Total characters: 1014
\n📊 Last 8 turns (8 messages):
  1. USER: What about feature engineering? Are there specific technique...
  2. ASSISTANT: Absolutely! For real estate: 1) Create price per square foot...
  3. USER: How do I evaluate my model's performance and avoid overfit

In [59]:
print("📏 Testing Truncation by Character/Word Limits")
print("=" * 45)

if GROQ_API_KEY != "your_groq_api_key_here":
    # Test character limit
    char_limits = [300, 600, 1000]
    for limit in char_limits:
        truncated = manager.truncate_by_length(max_chars=limit)
        actual_chars = sum(len(msg['content']) for msg in truncated)
        print(f"\\n📊 Character limit {limit}: {len(truncated)} messages, {actual_chars} actual chars")

    # Test word limit
    word_limits = [50, 100, 200]
    for limit in word_limits:
        truncated = manager.truncate_by_length(max_words=limit)
        actual_words = sum(len(msg['content'].split()) for msg in truncated)
        print(f"\\n📊 Word limit {limit}: {len(truncated)} messages, {actual_words} actual words")
else:
    print("⚠️  Please set your GROQ_API_KEY to test truncation!")


📏 Testing Truncation by Character/Word Limits
\n📊 Character limit 300: 0 messages, 0 actual chars
\n📊 Character limit 600: 2 messages, 363 actual chars
\n📊 Character limit 1000: 4 messages, 728 actual chars
\n📊 Word limit 50: 2 messages, 48 actual words
\n📊 Word limit 100: 3 messages, 95 actual words
\n📊 Word limit 200: 7 messages, 197 actual words


### 3. Demonstrating Summarization


In [60]:
print("📋 Testing Single Conversation Summarization")
print("=" * 45)

if GROQ_API_KEY != "your_groq_api_key_here":
    # Get current conversation
    current_conversation = manager.conversation_history
    
    print(f"Original conversation: {len(current_conversation)} messages")
    print(f"Total characters: {sum(len(msg['content']) for msg in current_conversation)}")
    
    print("\\n🤖 Generating summary with Groq API...")
    summary = manager.summarize_conversation(current_conversation)
    
    print(f"\\n📝 Summary ({len(summary)} characters):")
    print("-" * 40)
    print(summary)
    print("-" * 40)
else:
    print("⚠️  Please set your GROQ_API_KEY to test summarization!")


📋 Testing Single Conversation Summarization
Original conversation: 12 messages
Total characters: 2246
\n🤖 Generating summary with Groq API...
\n📝 Summary (813 characters):
----------------------------------------
Here's a concise summary of the conversation:

For a house price prediction project, use regression models (e.g., Linear Regression, Random Forest, XGBoost) and gather features like location, size, and local amenities. Handle missing values with imputation or dropping features, and consider feature engineering techniques like price per square foot and distance to city center. Evaluate model performance with metrics like RMSE, MAE, and R² score, and use techniques like cross-validation and regularization to avoid overfitting. Traditional machine learning often outperforms deep learning for tabular data, but consider deep learning if incorporating image data or complex feature interactions. To make the model interpretable, use techniques like feature importance, SHAP values, and

### 4. Demonstrating Periodic Summarization


In [61]:
print("⏰ Testing Periodic Summarization (Every 3 Runs)")
print("=" * 50)

if GROQ_API_KEY != "your_groq_api_key_here":
    # Reset manager for clean demo
    demo_manager = ConversationManager(GROQ_API_KEY)
    
    # Add initial conversation
    for role, content in sample_conversations[:4]:  # Add first 4 messages
        demo_manager.add_message(role, content)
    
    # Simulate multiple runs with periodic summarization
    additional_conversations = [
        ("user", "Can you help me with error handling in web scraping?"),
        ("assistant", "Absolutely! Use try-except blocks to handle requests exceptions, check response status codes, and implement retry logic for failed requests."),
        ("user", "What about handling different response formats like JSON vs HTML?"),
        ("assistant", "Great question! Check the Content-Type header first. For JSON, use response.json(). For HTML, use BeautifulSoup. Always validate the format before parsing."),
        ("user", "How do I store the scraped data efficiently?"),
        ("assistant", "For small datasets, use CSV with pandas. For larger datasets, consider SQLite or PostgreSQL. For real-time processing, use message queues like Redis."),
        ("user", "What about legal considerations when scraping?"),
        ("assistant", "Always check robots.txt, respect rate limits, read terms of service, consider API alternatives, and be mindful of copyright and personal data protection laws.")
    ]
    
    for run in range(1, 7):
        print(f"\\n🔄 Run #{run}")
        print("-" * 20)
        
        # Add new messages for this run
        if run <= len(additional_conversations):
            role, content = additional_conversations[run-1]
            demo_manager.add_message(role, content)
            print(f"Added: {role} - {content[:50]}...")
        
        # Show stats before summarization
        stats = demo_manager.get_conversation_length()
        print(f"Before: {stats['turns']} turns, {stats['characters']} chars")
        
        # Perform periodic summarization every 3 runs
        demo_manager.periodic_summarization(k_runs=3)
        
        # Show stats after summarization
        stats_after = demo_manager.get_conversation_length()
        print(f"After: {stats_after['turns']} turns, {stats_after['characters']} chars")
        
        if run % 3 == 0:
            print(f"✅ Summarization performed at run {run}")
            if demo_manager.summary:
                print(f"Summary preview: {demo_manager.summary[:100]}...")
else:
    print("⚠️  Please set your GROQ_API_KEY to test periodic summarization!")


⏰ Testing Periodic Summarization (Every 3 Runs)
\n🔄 Run #1
--------------------
Added: user - Can you help me with error handling in web scrapin...
Before: 5 turns, 843 chars
After: 5 turns, 843 chars
\n🔄 Run #2
--------------------
Added: assistant - Absolutely! Use try-except blocks to handle reques...
Before: 6 turns, 983 chars
After: 6 turns, 983 chars
\n🔄 Run #3
--------------------
Added: user - What about handling different response formats lik...
Before: 7 turns, 1048 chars

--- Performing periodic summarization (Run #3) ---
Conversation summarized and replaced. New summary length: 488 characters
After: 1 turns, 510 chars
✅ Summarization performed at run 3
Summary preview: Here's a concise summary of the conversation:

The user is starting a machine learning project to pr...
\n🔄 Run #4
--------------------
Added: assistant - Great question! Check the Content-Type header firs...
Before: 2 turns, 665 chars
After: 2 turns, 665 chars
\n🔄 Run #5
--------------------
Added: user - Ho

## Task 2: JSON Schema Classification & Information Extraction

1. Create a JSON schema to extract 5 details from chats related to information collection (e.g., name, email, phone, location, age).

2. Use OpenAI function calling with Groq API for structured outputs.

3. Demonstrate:
a. Parsing of at least 3 sample chats.
b. Validation of outputs against the schema.

In [62]:
# Additional imports for Task 2
import jsonschema
from jsonschema import validate, ValidationError
from typing import Dict, Any, List, Optional
import re

print("✅ Additional imports for Task 2 loaded!")


✅ Additional imports for Task 2 loaded!


## Information Extraction Schema




In [63]:
# Define JSON schema for personal information extraction
PERSONAL_INFO_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {
            "type": ["string", "null"],
            "description": "Full name of the person (first and last name)",
            "pattern": "^[A-Za-z\\s'-]+$"
        },
        "email": {
            "type": ["string", "null"],
            "description": "Email address",
            "pattern": "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
        },
        "phone": {
            "type": ["string", "null"],
            "description": "Phone number in any format",
            "pattern": "^[\\+]?[0-9\\s\\-\\(\\)\\.\\_]{7,20}$"
        },
        "location": {
            "type": ["string", "null"],
            "description": "Location (city, state, country, or address)"
        },
        "age": {
            "type": ["integer", "null"],
            "description": "Age in years",
            "minimum": 0,
            "maximum": 120
        }
    },
    "required": [],
    "additionalProperties": False
}

# Function definition for OpenAI function calling
EXTRACT_INFO_FUNCTION = {
    "name": "extract_personal_info",
    "description": "Extract personal information from chat conversations including name, email, phone, location, and age",
    "parameters": PERSONAL_INFO_SCHEMA
}

print("✅ JSON Schema and Function Definition Created!")
print("📋 Schema includes: name, email, phone, location, age")
print("🔧 Function ready for Groq API function calling")


✅ JSON Schema and Function Definition Created!
📋 Schema includes: name, email, phone, location, age
🔧 Function ready for Groq API function calling


## Information Extraction Class



In [64]:
class InformationExtractor:
    """
    Extracts personal information from chat conversations using Groq API function calling
    """
    
    def __init__(self, groq_api_key: str, model: str = "llama-3.1-8b-instant"):
        """
        Initialize the information extractor
        
        Args:
            groq_api_key: Groq API key
            model: Model to use for extraction
        """
        self.client = OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=groq_api_key
        )
        self.model = model
        self.schema = PERSONAL_INFO_SCHEMA
        
    def validate_extracted_info(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Validate extracted information against the JSON schema
        
        Args:
            data: Extracted information dictionary
            
        Returns:
            Validation result with status and errors if any
        """
        try:
            validate(instance=data, schema=self.schema)
            return {
                "valid": True,
                "errors": [],
                "data": data
            }
        except ValidationError as e:
            return {
                "valid": False,
                "errors": [str(e)],
                "data": data
            }
    
    def extract_information_from_chat(self, chat_messages: List[Dict[str, str]]) -> Dict[str, Any]:
        """
        Extract personal information from a chat conversation using function calling
        
        Args:
            chat_messages: List of chat messages with 'role' and 'content'
            
        Returns:
            Dictionary containing extracted information and validation status
        """
        # Format chat for processing
        conversation_text = "\n".join([
            f"{msg['role'].upper()}: {msg['content']}" 
            for msg in chat_messages
        ])
        
        system_prompt = """You are an AI assistant that extracts personal information from chat conversations.
Extract the following information if present in the conversation:
- name: Full name of the person
- email: Email address 
- phone: Phone number
- location: Location (city, state, country, or address)
- age: Age in years

If information is not mentioned or unclear, return null for that field.
Be accurate and only extract information that is explicitly stated."""
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Extract personal information from this conversation:\n\n{conversation_text}"}
                ],
                functions=[EXTRACT_INFO_FUNCTION],
                function_call={"name": "extract_personal_info"},
                temperature=0.1
            )
            
            # Parse function call result
            function_call = response.choices[0].message.function_call
            if function_call and function_call.name == "extract_personal_info":
                extracted_data = json.loads(function_call.arguments)
                
                # Validate the extracted data
                validation_result = self.validate_extracted_info(extracted_data)
                
                return {
                    "success": True,
                    "extracted_info": extracted_data,
                    "validation": validation_result,
                    "raw_response": function_call.arguments
                }
            else:
                return {
                    "success": False,
                    "error": "No function call in response",
                    "extracted_info": {},
                    "validation": {"valid": False, "errors": ["No function call"], "data": {}}
                }
                
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "extracted_info": {},
                "validation": {"valid": False, "errors": [str(e)], "data": {}}
            }
    
    def display_extraction_result(self, result: Dict[str, Any], chat_title: str = ""):
        """
        Display extraction results in a formatted way
        
        Args:
            result: Result from extract_information_from_chat
            chat_title: Optional title for the chat
        """
        print(f"\n{'='*50}")
        if chat_title:
            print(f"📋 Chat: {chat_title}")
        print(f"{'='*50}")
        
        if result["success"]:
            extracted = result["extracted_info"]
            validation = result["validation"]
            
            print(f"✅ Extraction Status: {'SUCCESS' if result['success'] else 'FAILED'}")
            print(f"✅ Validation Status: {'VALID' if validation['valid'] else 'INVALID'}")
            
            if not validation["valid"]:
                print(f"❌ Validation Errors: {', '.join(validation['errors'])}")
            
            print("\n📊 Extracted Information:")
            print("-" * 30)
            
            for field, value in extracted.items():
                status = "✅" if value is not None else "❌"
                display_value = value if value is not None else "Not found"
                print(f"{status} {field.capitalize()}: {display_value}")
                
        else:
            print(f"❌ Extraction Status: FAILED")
            print(f"❌ Error: {result.get('error', 'Unknown error')}")
        
        print(f"{'='*50}")

print("✅ InformationExtractor class loaded successfully!")


✅ InformationExtractor class loaded successfully!


### Sample Chat Conversations




In [65]:
# Sample chat conversations for testing information extraction
SAMPLE_CHATS = {
    "customer_support": [
        {"role": "assistant", "content": "Hello! Welcome to TechCorp customer support. How can I help you today?"},
        {"role": "user", "content": "Hi! I'm having trouble with my account. My name is Sarah Johnson and I can't log in."},
        {"role": "assistant", "content": "I'm sorry to hear that, Sarah. Can you please provide your email address so I can look up your account?"},
        {"role": "user", "content": "Sure, it's sarah.j.2024@gmail.com. I'm calling from Seattle, Washington."},
        {"role": "assistant", "content": "Thank you. I see your account. Can you also confirm your phone number for security purposes?"},
        {"role": "user", "content": "Yes, it's +1-206-555-0123. I'm 28 years old, by the way, if that helps with verification."},
        {"role": "assistant", "content": "Perfect! I can see your account now. Let me help you reset your password."}
    ],
    
    "job_application": [
        {"role": "assistant", "content": "Thank you for your interest in our Software Engineer position. Could you tell me a bit about yourself?"},
        {"role": "user", "content": "Absolutely! I'm Michael Chen, I'm 32 and I live in Austin, Texas."},
        {"role": "assistant", "content": "Great! What's the best way to contact you?"},
        {"role": "user", "content": "You can reach me at mchen.dev@hotmail.com or call me at 512-444-7890."},
        {"role": "assistant", "content": "Excellent. Can you tell me about your recent work experience?"},
        {"role": "user", "content": "I've been working as a full-stack developer for the past 5 years, mainly with React and Node.js."}
    ],
    
    "delivery_service": [
        {"role": "assistant", "content": "Hi there! I'm here to help with your delivery. What's your order number?"},
        {"role": "user", "content": "Hi! My order number is #DEL-45789. I'm wondering about the delivery time."},
        {"role": "assistant", "content": "Let me check that for you. Could you confirm your delivery address?"},
        {"role": "user", "content": "Yes, I'm at 123 Oak Street, Los Angeles, CA 90210. This is Emma Rodriguez speaking."},
        {"role": "assistant", "content": "Thank you Emma. And your contact number in case our driver needs to reach you?"},
        {"role": "user", "content": "It's 323-777-5555. I should mention I'm only 19, so please make sure to check ID if needed."},
        {"role": "assistant", "content": "Noted! Your delivery should arrive within the next 30 minutes."}
    ],
    
    "social_conversation": [
        {"role": "user", "content": "Hey! How's your day going?"},
        {"role": "assistant", "content": "Pretty good! Just working on some projects. What about you?"},
        {"role": "user", "content": "Same here! I'm just relaxing at home, maybe going to watch a movie later."},
        {"role": "assistant", "content": "That sounds nice! Any particular movie in mind?"},
        {"role": "user", "content": "Thinking about that new sci-fi film everyone's talking about. Have you seen it?"},
        {"role": "assistant", "content": "Not yet, but I've heard great things about it!"}
    ]
}

print("✅ Sample chat conversations created!")
print(f"📋 Total chats: {len(SAMPLE_CHATS)}")
for chat_name, messages in SAMPLE_CHATS.items():
    print(f"  - {chat_name}: {len(messages)} messages")


✅ Sample chat conversations created!
📋 Total chats: 4
  - customer_support: 7 messages
  - job_application: 6 messages
  - delivery_service: 7 messages
  - social_conversation: 6 messages


### Demonstration: Information Extraction and Validation



In [69]:
# Initialize the Information Extractor
print("🚀 Initializing Information Extractor with Groq API...")
extractor = InformationExtractor(GROQ_API_KEY)
print("✅ Information Extractor initialized successfully!")
print(f"🔧 Using model: {extractor.model}")

# Test schema validation with sample data
print("\n🧪 Testing schema validation...")

# Test 1: Valid data example (should pass)
valid_data = {
    "name": "John Doe",
    "email": "john.doe@example.com", 
    "phone": "555-123-4567",  # This now works with updated regex
    "location": "New York, NY",
    "age": 30
}

validation_result = extractor.validate_extracted_info(valid_data)
print(f"✅ Valid data test: {validation_result['valid']} (Expected: True)")

# Test 2: Invalid data example - bad email (should fail)
invalid_data = {
    "name": "Jane Doe",
    "email": "invalid-email",  # Invalid email format
    "phone": "555-987-6543",   # Valid phone with updated regex
    "location": "Boston, MA",
    "age": 25
}

validation_result = extractor.validate_extracted_info(invalid_data)
print(f"❌ Invalid email test: {validation_result['valid']} (Expected: False)")
if not validation_result['valid']:
    print(f"   Reason: Email validation failed - {validation_result['errors'][0].split(':')[1].strip() if ':' in validation_result['errors'][0] else 'Invalid email format'}")

# Test 3: Invalid phone number (should fail with old restrictive pattern)
invalid_phone_data = {
    "name": "Test User",
    "email": "test@example.com",
    "phone": "ABC-123-DEFG",  # Invalid phone format
    "location": "Test City",
    "age": 30
}

validation_result = extractor.validate_extracted_info(invalid_phone_data)
print(f"❌ Invalid phone test: {validation_result['valid']} (Expected: False)")
if not validation_result['valid']:
    print(f"   Reason: Phone validation failed - contains non-numeric characters")


🚀 Initializing Information Extractor with Groq API...
✅ Information Extractor initialized successfully!
🔧 Using model: llama-3.1-8b-instant

🧪 Testing schema validation...
✅ Valid data test: True (Expected: True)
❌ Invalid email test: False (Expected: False)
   Reason: Email validation failed - {'type'
❌ Invalid phone test: False (Expected: False)
   Reason: Phone validation failed - contains non-numeric characters


In [71]:
# Extract information from all sample chats
print("🔍 Starting Information Extraction from Sample Chats")
print("=" * 60)

extraction_results = {}

for chat_name, messages in SAMPLE_CHATS.items():
    print(f"\n🗣️  Processing: {chat_name.replace('_', ' ').title()}")
    print("-" * 40)
    
    # Show the conversation
    print("📝 Conversation:")
    for i, msg in enumerate(messages[:3], 1):  # Show first 3 messages
        content_preview = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
        print(f"  {i}. {msg['role'].upper()}: {content_preview}")
    
    if len(messages) > 3:
        print(f"  ... and {len(messages) - 3} more messages")
    
    print("\n🤖 Extracting information with Groq API...")
    
    # Extract information
    result = extractor.extract_information_from_chat(messages)
    extraction_results[chat_name] = result
    
    # Display results
    extractor.display_extraction_result(result, chat_name.replace('_', ' ').title())
    
    # Add a small delay to be respectful to the API
    import time
    time.sleep(1)

# Summary statistics
print(f"\n📊 EXTRACTION SUMMARY")
print("=" * 60)

successful_extractions = sum(1 for result in extraction_results.values() if result['success'])
valid_extractions = sum(1 for result in extraction_results.values() 
                      if result['success'] and result['validation']['valid'])

print(f"✅ Successful extractions: {successful_extractions}/{len(SAMPLE_CHATS)}")
print(f"✅ Valid extractions: {valid_extractions}/{len(SAMPLE_CHATS)}")

# Field extraction statistics
field_stats = {"name": 0, "email": 0, "phone": 0, "location": 0, "age": 0}

for result in extraction_results.values():
    if result['success']:
        extracted = result['extracted_info']
        for field, value in extracted.items():
            if value is not None:
                field_stats[field] += 1

print(f"\n📈 Field Extraction Statistics:")
for field, count in field_stats.items():
    percentage = (count / len(SAMPLE_CHATS)) * 100
    print(f"  {field.capitalize()}: {count}/{len(SAMPLE_CHATS)} chats ({percentage:.1f}%)")


🔍 Starting Information Extraction from Sample Chats

🗣️  Processing: Customer Support
----------------------------------------
📝 Conversation:
  1. ASSISTANT: Hello! Welcome to TechCorp customer support. How can I help you today?
  2. USER: Hi! I'm having trouble with my account. My name is Sarah Johnson and I can't log...
  3. ASSISTANT: I'm sorry to hear that, Sarah. Can you please provide your email address so I ca...
  ... and 4 more messages

🤖 Extracting information with Groq API...

📋 Chat: Customer Support
✅ Extraction Status: SUCCESS
✅ Validation Status: VALID

📊 Extracted Information:
------------------------------
✅ Age: 28
✅ Email: sarah.j.2024@gmail.com
✅ Location: Seattle, Washington
✅ Name: Sarah Johnson
✅ Phone: +1-206-555-0123

🗣️  Processing: Job Application
----------------------------------------
📝 Conversation:
  1. ASSISTANT: Thank you for your interest in our Software Engineer position. Could you tell me...
  2. USER: Absolutely! I'm Michael Chen, I'm 32 and I li